#Pytorch Pipeline

#Importing Packages

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import copy

#Data Preprocessing

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081))
])
train_dataset = torchvision.datasets.MNIST(root='./data',download=True,train=True,transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data',download=True,train=False,transform=transform)

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=1000,shuffle=False)

#Model

In [ ]:
class MNISTClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.layers = nn.Sequential(
        nn.Linear(28*28,128),
        nn.ReLU(),
        nn.Linear(128,10)
    )
  def forward(self,x):
    x = self.flatten(x)
    x = self.layers(x)

    return x

#Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNISTClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)
def train_epoch(model,train_loaderr,loss_function,optimizer,device):
  model.train()
  running_loss = 0.0
  correct =0
  total = 0


  for batch_indx,(data,targets) in enumerate(train_loader):
    data,targets = data.to(device),targets.to(device)
    optimizer.zero_grad()
    outputs = model(data)
    loss = loss_function(outputs,targets)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    _,predicted = outputs.max(1)
    total += targets.size(0)
    correct += predicted.eq(targets).sum().item()

    if batch_indx % 100 == 0 and batch_indx > 0:

      avg_loss = running_loss / 100
      accuracy = 100. * correct / total
      print(f"Batch : {batch_indx*64}| {len(train_loader.dataset)} " f"Loss : {avg_loss:.3f} | Accuracy : {accuracy:.1f}")
      running_loss = 0


#Evaluation

In [ ]:
def evaluate(model,test_loader,device):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for data,targets in test_loader:
      inputs,targets = data.to(device),targets.to(device)
      outputs = model(inputs)
      _,predicted = outputs.max(1)
      total += targets.size(0)
      correct += predicted.eq(targets).sum().item()
  return 100. * correct / total

#Test

In [ ]:
epochs =5
best_model = None
best_accuracy = 0.0
best_epoch = 0
for epoch in range(epochs):
  print(f"Epoch : {epoch+1}")
  train_epoch(model,train_loader,loss_function,optimizer,device)
  accuracy = evaluate(model,test_loader,device)
  if accuracy > best_accuracy:
    best_epoch = epoch+1
    best_accuracy = accuracy
    best_model = copy.deepcopy(model)
  print(f"Test Accuracy :{accuracy:.2f}")
print(f"The Best Model Accuracy is {best_accuracy} at Epoch {best_epoch}")

Epoch : 1
Batch : 6400| 60000 Loss : 0.647 | Accuracy : 81.7
Batch : 12800| 60000 Loss : 0.316 | Accuracy : 86.2
Batch : 19200| 60000 Loss : 0.285 | Accuracy : 88.0
Batch : 25600| 60000 Loss : 0.235 | Accuracy : 89.3
Batch : 32000| 60000 Loss : 0.206 | Accuracy : 90.2
Batch : 38400| 60000 Loss : 0.185 | Accuracy : 90.9
Batch : 44800| 60000 Loss : 0.171 | Accuracy : 91.5
Batch : 51200| 60000 Loss : 0.153 | Accuracy : 92.0
Batch : 57600| 60000 Loss : 0.161 | Accuracy : 92.4
Test Accuracy :95.84
Epoch : 2
Batch : 6400| 60000 Loss : 0.140 | Accuracy : 96.0
Batch : 12800| 60000 Loss : 0.115 | Accuracy : 96.3
Batch : 19200| 60000 Loss : 0.110 | Accuracy : 96.3
Batch : 25600| 60000 Loss : 0.098 | Accuracy : 96.5
Batch : 32000| 60000 Loss : 0.125 | Accuracy : 96.5
Batch : 38400| 60000 Loss : 0.108 | Accuracy : 96.5
Batch : 44800| 60000 Loss : 0.100 | Accuracy : 96.6
Batch : 51200| 60000 Loss : 0.108 | Accuracy : 96.6
Batch : 57600| 60000 Loss : 0.097 | Accuracy : 96.7
Test Accuracy :97.05
Epoc